# Scalability analysis of the Declare conformance checker

Reads the benchmark CSVs produced by `evaluation/benchmark_scalability.py` and produces the
thesis figures and tables. Experiments (protocol: single-threaded, fixed sampling seed,
warm-up pass, `time.perf_counter`, median over repetitions):

* **E1** runtime vs. log size (#cases), log-log fit for the empirical scaling exponent
* **E2** per-case runtime vs. trace length — natural range from the RTFM log plus
  synthetically lengthened traces (event sequence replicated $k\times$)
* **E3** runtime vs. number of constraints in the model
* **E4** per-rule runtime vs. the rule complexity measure
  $C(r) = w_{template} + n_{clauses} + n_{attributes} + t_{time}$ (see `evaluation/complexity.py`)
* **E5** peak memory vs. log size

Regenerate the inputs through the safe launcher (tmux + timestamped log + `--resume`):
```
./run_benchmark_safe.sh --reps 3 -- --e2-cases 5000 --e3-cases 5000 --e4-cases 5000
```
See `evaluation/README.md` for the sampling protocol — which experiments run on the full
150,370-case log and why the others subsample.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "evaluation" else Path.cwd()
BENCH_DIR = REPO_ROOT / "evaluation" / "data" / "benchmarks"
FIG_DIR = REPO_ROOT / "evaluation" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


def load(name, sample_col=None):
    """Read a benchmark CSV, keeping measurements from a single sample size.

    E2–E4 run on a subsample whose size is a run parameter. Resuming with a
    different `--eN-cases` appends measurements at the new size alongside the old
    ones — they are distinct units, so `--resume` re-measures them by design.
    Aggregating over both would silently average two sample sizes, so keep the
    largest and report what was dropped.
    """
    df = pd.read_csv(BENCH_DIR / name)
    if sample_col and sample_col in df.columns and df[sample_col].nunique() > 1:
        keep = df[sample_col].max()
        dropped = sorted(set(df[sample_col].unique()) - {keep})
        print(f"{name}: sample sizes {dropped + [keep]} present; keeping {keep}")
        df = df[df[sample_col] == keep]
    return df


e1 = load("e1_log_size.csv")
e2n = load("e2_trace_length_natural.csv", "n_sample_cases")
e2s = load("e2_trace_length_synthetic.csv", "n_sample_cases")
e3 = load("e3_model_size.csv", "n_cases")
e4 = load("e4_rule_complexity.csv", "n_cases")
e5 = load("e5_memory.csv")
machine = json.loads((BENCH_DIR / "machine_info.json").read_text())

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

def save(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", bbox_inches="tight")
    print(f"saved {FIG_DIR / name}.pdf/.png")

## E1 — Runtime vs. log size (and E5 memory)

Median wall time over repetitions per sample size, IQR as error bars, log-log axes. The fitted
slope of $\log t$ on $\log n$ is the empirical scaling exponent; per-case independence of the
checker predicts $\approx 1$ (linear scaling). Throughput (cases/s) should correspondingly stay
flat. The second panel shows `tracemalloc` peak memory from E5 (measured in a separate
instrumented run; its wall times are not comparable with E1).

In [ ]:
agg = e1.groupby("n_cases")["wall_s"].agg(
    median="median", q1=lambda s: s.quantile(0.25), q3=lambda s: s.quantile(0.75)
)
slope, intercept, r, *_ = stats.linregress(np.log(agg.index), np.log(agg["median"]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.8))
ax1.errorbar(agg.index, agg["median"], yerr=[agg["median"] - agg["q1"], agg["q3"] - agg["median"]],
             fmt="o-", capsize=3)
xs = np.array([agg.index.min(), agg.index.max()])
ax1.plot(xs, np.exp(intercept) * xs**slope, "--", color="gray",
         label=f"fit: $t \\propto n^{{{slope:.2f}}}$  ($R^2$={r**2:.3f})")
ax1.set(xscale="log", yscale="log", xlabel="cases", ylabel="wall time [s]",
        title="E1: runtime vs. log size")
ax1.legend()

ax2.plot(e5["n_cases"], e5["tracemalloc_peak_mb"], "s-", color="tab:orange")
ax2.set(xscale="log", xlabel="cases", ylabel="tracemalloc peak [MB]",
        title="E5: peak memory vs. log size")
fig.tight_layout()
save(fig, "scalability_log_size")

print(f"scaling exponent: {slope:.3f}")
throughput = e1.groupby("n_cases")["cases_per_s"].median()
print("median throughput (cases/s) per size:")
print(throughput.round(1).to_string())

## E2 — Runtime vs. trace length

Left: per-case wall time over the log's natural trace lengths (2–20 events), median per length
with IQR band. Right: synthetically lengthened traces (whole trace checked as one unit, no
segmentation) on log-log axes with a fitted exponent — this exposes the checker's per-trace
scaling (activation×target candidate loops, per-evaluation snapshot construction) beyond the
short traces the RTFM log naturally offers.

In [ ]:
nat = e2n.groupby("n_events")["wall_s"].agg(
    median="median", q1=lambda s: s.quantile(0.25), q3=lambda s: s.quantile(0.75), n="size"
)
nat = nat[nat["n"] >= 10]  # drop sparsely populated lengths

syn = e2s.groupby("n_events")["wall_s"].agg(
    median="median", q1=lambda s: s.quantile(0.25), q3=lambda s: s.quantile(0.75)
)
s_slope, s_int, s_r, *_ = stats.linregress(np.log(syn.index), np.log(syn["median"]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.8))
ax1.plot(nat.index, nat["median"] * 1e3, "o-")
ax1.fill_between(nat.index, nat["q1"] * 1e3, nat["q3"] * 1e3, alpha=0.25)
ax1.set(xlabel="events per trace", ylabel="per-case wall time [ms]",
        title="E2: natural trace lengths")

ax2.errorbar(syn.index, syn["median"], yerr=[syn["median"] - syn["q1"], syn["q3"] - syn["median"]],
             fmt="o", capsize=3)
xs = np.array([syn.index.min(), syn.index.max()])
ax2.plot(xs, np.exp(s_int) * xs**s_slope, "--", color="gray",
         label=f"fit: $t \\propto m^{{{s_slope:.2f}}}$  ($R^2$={s_r**2:.3f})")
ax2.set(xscale="log", yscale="log", xlabel="events per trace", ylabel="per-trace wall time [s]",
        title="E2: synthetic lengthening")
ax2.legend()
fig.tight_layout()
save(fig, "scalability_trace_length")
print(f"trace-length scaling exponent (synthetic): {s_slope:.3f}")

## E3 — Runtime vs. number of constraints

Fixed case sample, model truncated to its first $k$ constraints. Constraints are evaluated
independently, so runtime should grow linearly in $k$; the per-constraint increments differ
because the RTFM constraints differ in complexity (that heterogeneity is exactly what E4
quantifies).

In [ ]:
agg3 = e3.groupby("k_constraints")["wall_s"].agg(
    median="median", q1=lambda s: s.quantile(0.25), q3=lambda s: s.quantile(0.75)
)
l_slope, l_int, l_r, *_ = stats.linregress(agg3.index, agg3["median"])

fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.errorbar(agg3.index, agg3["median"],
            yerr=[agg3["median"] - agg3["q1"], agg3["q3"] - agg3["median"]], fmt="o", capsize=3)
ax.plot(agg3.index, l_int + l_slope * agg3.index, "--", color="gray",
        label=f"linear fit: {l_slope:.2f} s/constraint  ($R^2$={l_r**2:.3f})")
ax.set(xlabel="constraints in model", ylabel="wall time [s]",
       title="E3: runtime vs. model size", xticks=agg3.index)
ax.legend()
fig.tight_layout()
save(fig, "scalability_model_size")

## E4 — Runtime vs. rule complexity

Each point is one rule (real `rtfm.yaml` model + synthetic sweep variants), x = complexity
score $C(r)$, y = median per-evaluation runtime. Reported: Spearman rank correlation between
score and runtime, and an OLS decomposition of per-evaluation runtime on the individual
components — the marginal cost in µs of one template grade, one clause, one attribute.
(`t_time` is carried in the score but excluded from the regression: only one measured rule has
a time window, which cannot identify a coefficient.)

In [ ]:
per_rule = (
    e4.groupby(["model", "rule_id", "template", "w_template", "n_clauses", "n_attributes",
                "t_time", "complexity"])
    ["mean_us_per_eval"].median().reset_index()
)
rho, rho_p = stats.spearmanr(per_rule["complexity"], per_rule["mean_us_per_eval"])

X = per_rule[["w_template", "n_clauses", "n_attributes"]].to_numpy(dtype=float)
X = np.column_stack([np.ones(len(X)), X])
coef, *_ = np.linalg.lstsq(X, per_rule["mean_us_per_eval"].to_numpy(), rcond=None)
pred = X @ coef
ss_res = np.sum((per_rule["mean_us_per_eval"] - pred) ** 2)
ss_tot = np.sum((per_rule["mean_us_per_eval"] - per_rule["mean_us_per_eval"].mean()) ** 2)
r2 = 1 - ss_res / ss_tot

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.8))
for model_name, marker in (("rtfm", "o"), ("rtfm_synthetic", "^")):
    sub = per_rule[per_rule["model"] == model_name]
    ax1.scatter(sub["complexity"], sub["mean_us_per_eval"], marker=marker, label=model_name)
ax1.set(xlabel="complexity score $C(r)$", ylabel="per-evaluation runtime [\u00b5s]",
        title=f"E4: runtime vs. complexity  (Spearman $\\rho$={rho:.2f}, p={rho_p:.1e})")
ax1.legend()

labels = ["intercept", "template grade", "clause", "attribute"]
ax2.bar(labels[1:], coef[1:], color="tab:blue")
ax2.set(ylabel="marginal cost [\u00b5s per evaluation]",
        title=f"OLS decomposition  ($R^2$={r2:.2f})")
fig.tight_layout()
save(fig, "scalability_rule_complexity")

print(f"Spearman rho={rho:.3f} (p={rho_p:.2e}); OLS coefficients [us]:")
for name, c in zip(labels, coef):
    print(f"  {name:>14}: {c:8.1f}")

## Tables

Complexity breakdown and measured per-evaluation runtime for the 10 RTFM rules (thesis table),
and the benchmark environment.

In [ ]:
table = per_rule[per_rule["model"] == "rtfm"][
    ["rule_id", "template", "w_template", "n_clauses", "n_attributes", "t_time",
     "complexity", "mean_us_per_eval"]
].sort_values("rule_id").reset_index(drop=True)
table["mean_us_per_eval"] = table["mean_us_per_eval"].round(1)
table.to_csv(FIG_DIR / "table_rule_complexity.csv", index=False)
display(table)

env = pd.Series({
    "CPU": machine.get("cpu_model", "?"),
    "cores": machine.get("cpu_count"),
    "RAM [GB]": round(machine["ram_total_gb"], 1) if "ram_total_gb" in machine else "?",
    "platform": machine.get("platform"),
    "Python": machine.get("python", "?").split()[0],
    "pandas": machine.get("pandas"),
    "git commit": str(machine.get("git_commit", ""))[:12],
    "run date (UTC)": str(machine.get("timestamp_utc", ""))[:19],
}, name="benchmark environment")
display(env.to_frame())